# 05 · Prepare Documents for RAG


In [1]:
%pip install -q langchain-text-splitters pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 25.0 MB/s eta 0:00:00


## Create a sample source document


In [2]:
from pathlib import Path

docs_dir = Path("sample_docs")
docs_dir.mkdir(exist_ok=True)
sample = docs_dir / "student_services.txt"
sample.write_text(
    "Library Services\n"
    "Students can borrow books using their campus identity card. Renewals are available through the library portal.\n\n"
    "IT Support\n"
    "For account access problems, first check your username and keyboard layout. Use the password recovery workflow if needed.\n\n"
    "Sports Facilities\n"
    "Court bookings are made through the recreation desk. Users should cancel reservations they can no longer attend.\n",
    encoding="utf-8",
)
text = sample.read_text(encoding="utf-8")
print(text)

Library Services
Students can borrow books using their campus identity card. Renewals are available through the library portal.

IT Support
For account access problems, first check your username and keyboard layout. Use the password recovery workflow if needed.

Sports Facilities
Court bookings are made through the recreation desk. Users should cancel reservations they can no longer attend.



## Compare chunking approaches


In [3]:
def fixed_chunks(text, size=180, overlap=30):
    if size <= 0 or overlap < 0 or overlap >= size:
        raise ValueError("Require size > overlap >= 0")
    result, start = [], 0
    while start < len(text):
        result.append(text[start:start + size])
        start += size - overlap
    return result

from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=180,
    chunk_overlap=30,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks_fixed = fixed_chunks(text)
chunks_recursive = recursive_splitter.split_text(text)
print("Fixed-size chunks:", len(chunks_fixed))
print("Recursive chunks:", len(chunks_recursive))
for i, chunk in enumerate(chunks_recursive, start=1):
    print(f"\n--- chunk {i} ({len(chunk)} chars) ---\n{chunk}")

Fixed-size chunks: 3
Recursive chunks: 3

--- chunk 1 (127 chars) ---
Library Services
Students can borrow books using their campus identity card. Renewals are available through the library portal.

--- chunk 2 (132 chars) ---
IT Support
For account access problems, first check your username and keyboard layout. Use the password recovery workflow if needed.

--- chunk 3 (130 chars) ---
Sports Facilities
Court bookings are made through the recreation desk. Users should cancel reservations they can no longer attend.


## Attach metadata to each chunk


In [4]:
chunk_records = [
    {
        "id": f"{sample.stem}_{i:03d}",
        "text": chunk,
        "metadata": {"source": sample.name, "chunk_index": i},
    }
    for i, chunk in enumerate(chunks_recursive)
]
chunk_records[:2]

[{'id': 'student_services_000',
  'text': 'Library Services\nStudents can borrow books using their campus identity card. Renewals are available through the library portal.',
  'metadata': {'source': 'student_services.txt', 'chunk_index': 0}},
 {'id': 'student_services_001',
  'text': 'IT Support\nFor account access problems, first check your username and keyboard layout. Use the password recovery workflow if needed.',
  'metadata': {'source': 'student_services.txt', 'chunk_index': 1}}]

## Load a PDF you provide (optional)


In [5]:
# Point this at a PDF file that you own or are allowed to process.
from pypdf import PdfReader

def read_pdf(path):
    reader = PdfReader(path)
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text() or ""
        if page_text.strip():
            pages.append({"page": page_number, "text": page_text})
    return pages

# Example:
# pages = read_pdf("sample_docs/your_document.pdf")
# print("Pages with text:", len(pages))
# print(pages[0] if pages else "No extractable text found.")